# Day 079 — Solution: A Minimal Agent from Scratch

In [ ]:
_SRC = '"""simple_agent.py — Day 079: What Is an Agent?\n\nA minimal agent, built from scratch. An *agent* is an LLM wrapped in a loop\nthat can call tools and decide for itself when it is finished. That is the one\ndifference from a *pipeline*: a pipeline runs a fixed sequence of steps with no\ndecisions; an agent chooses its next action each turn until the task is done.\n\nPieces (each introduced on Day 079):\n  safe_calculate           - evaluate arithmetic safely (no eval)\n  DEFAULT_TOOLS            - a small tool registry (calculator, word_count)\n  build_tool_descriptions  - render the registry as prompt text\n  safe_parse_json          - tolerant JSON parser for messy LLM output\n  parse_action             - turn LLM text into an action dict (never raises)\n  execute_tool             - run one tool from the registry\n  call_llm                 - Ollama wrapper with an llm_fn injection point\n  build_agent_prompt       - assemble the messages for one step\n  run_agent                - the agent loop (bounded by max_iterations)\n  SimpleAgent              - a stateful agent binding tools + llm_fn\n\nSetup:\n    pip install ollama\n    ollama pull llama3.2\n"""\nimport ast\nimport json\nimport operator\n\n# ── a safe calculator tool (no eval) ─────────────────────────────────────────\n_OPS = {\n    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,\n    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,\n    ast.USub: operator.neg, ast.UAdd: operator.pos,\n}\n\n\ndef _eval_node(node):\n    """Recursively evaluate an arithmetic AST node. Raises on anything unsafe."""\n    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):\n        return node.value\n    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:\n        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))\n    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:\n        return _OPS[type(node.op)](_eval_node(node.operand))\n    raise ValueError("unsupported expression")\n\n\ndef safe_calculate(expression):\n    """Evaluate a basic arithmetic expression without eval().\n\n    Supports + - * / ** % and parentheses. Anything else (names, calls,\n    attribute access) raises ValueError. This is the safe way to give an\n    agent a calculator: never eval() untrusted model output.\n    """\n    tree = ast.parse(expression, mode="eval")\n    return _eval_node(tree.body)\n\n\n# ── the tool registry ────────────────────────────────────────────────────────\n# A tool = {description, parameters, fn}. fn takes an args dict, returns a str.\nDEFAULT_TOOLS = {\n    "calculator": {\n        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",\n        "parameters": {"expression": "string - the arithmetic to evaluate"},\n        "fn": lambda args: str(safe_calculate(args["expression"])),\n    },\n    "word_count": {\n        "description": "Count the words in a piece of text.",\n        "parameters": {"text": "string - the text to count words in"},\n        "fn": lambda args: str(len(str(args["text"]).split())),\n    },\n}\n\n\ndef build_tool_descriptions(tools):\n    """Render a tool registry as a text block for the prompt."""\n    lines = []\n    for name, spec in tools.items():\n        params = ", ".join(spec.get("parameters", {}))\n        lines.append("- " + name + "(" + params + "): " + spec["description"])\n    return "\\n".join(lines)\n\n# ── parsing messy LLM output ──────────────────────────────────────────────────\ndef safe_parse_json(text):\n    """Extract and parse the first JSON object from messy LLM output.\n\n    LLMs wrap JSON in markdown fences or prose. Instead of fighting that,\n    slice from the first \'{\' to the last \'}\' and parse that. Returns a dict,\n    or None if no valid JSON object is present.\n    """\n    start, end = text.find("{"), text.rfind("}")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, dict) else None\n\n\ndef parse_action(text):\n    """Turn raw LLM output into an action dict. NEVER raises.\n\n    Returns one of:\n      {"type": "tool",   "tool": name, "args": {...}}\n      {"type": "finish", "answer": str}\n    If the text is not a valid tool call, it falls back to a finish action\n    holding the raw text - so a badly-formatted model reply still terminates\n    the loop instead of crashing it.\n    """\n    data = safe_parse_json(text)\n    if not isinstance(data, dict):\n        return {"type": "finish", "answer": text.strip()}\n    tool = data.get("tool")\n    if tool and tool != "finish":\n        return {"type": "tool", "tool": tool, "args": data.get("args", {})}\n    return {"type": "finish", "answer": data.get("answer", text.strip())}\n\n# ── executing tools + calling the model ──────────────────────────────────────\ndef execute_tool(action, tools):\n    """Run one tool action against the registry. Returns a result string.\n\n    Never raises: an unknown tool or a tool error is returned as text so the\n    agent can read it and recover on its next turn.\n    """\n    name = action.get("tool")\n    if name not in tools:\n        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)\n    try:\n        return str(tools[name]["fn"](action.get("args", {})))\n    except Exception as exc:\n        return "Error running " + str(name) + ": " + str(exc)\n\n\ndef call_llm(messages, llm_fn=None):\n    """Call the chat model. Inject llm_fn(messages) -> str for testing.\n\n    llm_fn=None uses Ollama (llama3.2). A mock llm_fn lets the whole agent\n    run offline with no model - which is how the tests drive the loop.\n    """\n    if llm_fn is not None:\n        return llm_fn(messages)\n    import ollama\n    resp = ollama.chat(model="llama3.2", messages=messages)\n    return resp["message"]["content"]\n\n# ── the agent loop ────────────────────────────────────────────────────────────\ndef build_agent_prompt(task, tools, history):\n    """Build the [system, user] messages for one step of the loop."""\n    system = "\\n".join([\n        "You are a tool-using agent. Solve the task by choosing ONE action at "\n        "a time, returned as a single JSON object.",\n        "",\n        "Available tools:",\n        build_tool_descriptions(tools),\n        "",\n        "To use a tool, reply with exactly:",\n        \'{"tool": "<name>", "args": {...}}\',\n        "When you know the final answer, reply with exactly:",\n        \'{"tool": "finish", "answer": "<answer>"}\',\n        "",\n        "Reply with only the JSON object, nothing else.",\n    ])\n    lines = ["Task: " + str(task)]\n    for step in history:\n        lines.append("You called: " + json.dumps(step["action"]))\n        lines.append("Result: " + str(step["result"]))\n    lines.append("What is your next action?")\n    user = "\\n".join(lines)\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": user}]\n\n\ndef run_agent(task, tools=None, llm_fn=None, max_iterations=10):\n    """Run the agent loop until it finishes or hits max_iterations.\n\n    The loop is the whole idea of an agent: build a prompt from the task and\n    what has happened so far, ask the model for one action, run it, repeat.\n    max_iterations is the safeguard - without it a confused model could loop\n    forever. Returns:\n      {"answer": str, "steps": [...], "iterations": int, "stopped": bool}\n    stopped is True if the loop ran out of iterations without finishing.\n    """\n    if tools is None:\n        tools = DEFAULT_TOOLS\n    history = []\n    for i in range(max_iterations):\n        messages = build_agent_prompt(task, tools, history)\n        response = call_llm(messages, llm_fn=llm_fn)\n        action = parse_action(response)\n        if action["type"] == "finish":\n            return {"answer": action["answer"], "steps": history,\n                    "iterations": i + 1, "stopped": False}\n        result = execute_tool(action, tools)\n        history.append({"action": action, "result": result})\n    return {"answer": "Stopped: reached max_iterations without finishing.",\n            "steps": history, "iterations": max_iterations, "stopped": True}\n\n# ── the agent as a class ──────────────────────────────────────────────────────\nclass SimpleAgent:\n    """A minimal tool-using agent.\n\n    Binds a tool registry and an optional llm_fn at construction, then runs\n    tasks through run_agent and keeps a history of every run.\n\n    Example::\n\n        agent = SimpleAgent(llm_fn=my_llm_fn)\n        result = agent.run("What is 2 + 2?")\n        print(result["answer"])\n    """\n\n    def __init__(self, tools=None, llm_fn=None, max_iterations=10):\n        # copy so add_tool never mutates the shared DEFAULT_TOOLS global\n        self.tools = dict(DEFAULT_TOOLS if tools is None else tools)\n        self._llm_fn = llm_fn\n        self.max_iterations = max_iterations\n        self._history = []\n\n    def add_tool(self, name, description, fn, parameters=None):\n        """Register a new tool. fn takes an args dict and returns a result."""\n        self.tools[name] = {"description": description,\n                            "parameters": parameters or {},\n                            "fn": fn}\n        return self\n\n    def run(self, task):\n        """Run one task through the agent loop. Returns the result dict."""\n        result = run_agent(task, tools=self.tools, llm_fn=self._llm_fn,\n                           max_iterations=self.max_iterations)\n        self._history.append({"task": task, "result": result})\n        return result\n\n    def history(self):\n        """Return a copy of the run history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear the run history in place."""\n        self._history.clear()\n'
from pathlib import Path
Path('simple_agent.py').write_text(_SRC, encoding='utf-8')
print('simple_agent.py written.')

In [ ]:

from simple_agent import (
    safe_calculate, DEFAULT_TOOLS, build_tool_descriptions,
    safe_parse_json, parse_action, execute_tool, call_llm,
    build_agent_prompt, run_agent, SimpleAgent,
)

def _make_mock_llm(script):
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn

# 1. safe_calculate (no eval)
assert safe_calculate('2 * (3 + 4)') == 14
try:
    safe_calculate('__import__("os")'); raise AssertionError('should have raised')
except ValueError:
    pass
print("✅ safe_calculate")

# 2. tools + descriptions
assert DEFAULT_TOOLS['calculator']['fn']({'expression': '6*7'}) == '42'
assert 'calculator' in build_tool_descriptions(DEFAULT_TOOLS)
print("✅ DEFAULT_TOOLS + build_tool_descriptions")

# 3. parsing
assert safe_parse_json('prefix {"tool": "finish", "answer": "x"} suffix')['answer'] == 'x'
assert safe_parse_json('nope') is None
assert parse_action('plain answer')['type'] == 'finish'
assert parse_action('{"tool": "calculator", "args": {"expression": "1+1"}}')['type'] == 'tool'
print("✅ safe_parse_json + parse_action")

# 4. execute_tool
assert execute_tool({'tool': 'calculator', 'args': {'expression': '2+2'}}, DEFAULT_TOOLS) == '4'
assert 'unknown tool' in execute_tool({'tool': 'zzz', 'args': {}}, DEFAULT_TOOLS).lower()
assert 'error' in execute_tool({'tool': 'calculator', 'args': {}}, DEFAULT_TOOLS).lower()
print("✅ execute_tool")

# 5. call_llm injection
assert call_llm([{'role': 'user', 'content': 'hi'}], llm_fn=lambda m: 'MOCK') == 'MOCK'
print("✅ call_llm (llm_fn injection)")

# 6. run_agent: tool call then finish
script = ['{"tool": "calculator", "args": {"expression": "2+2"}}',
          '{"tool": "finish", "answer": "It is 4."}']
out = run_agent('what is 2+2', DEFAULT_TOOLS, llm_fn=_make_mock_llm(script))
assert out['answer'] == 'It is 4.' and out['stopped'] is False
assert len(out['steps']) == 1 and out['steps'][0]['result'] == '4'
print("✅ run_agent (loop: tool -> finish)")

# 7. max_iterations safeguard
never = _make_mock_llm(['{"tool": "calculator", "args": {"expression": "1+1"}}'])
loop = run_agent('x', DEFAULT_TOOLS, llm_fn=never, max_iterations=3)
assert loop['stopped'] is True and loop['iterations'] == 3
print("✅ run_agent (max_iterations stops runaway loop)")

# 8. SimpleAgent
agent = SimpleAgent(llm_fn=_make_mock_llm(script))
assert agent.run('2+2?')['answer'] == 'It is 4.'
agent.add_tool('shout', 'Uppercase.', lambda args: str(args['text']).upper(), {'text': 'string'})
assert 'shout' in agent.tools and 'shout' not in DEFAULT_TOOLS
assert len(agent.history()) == 1
agent.history().clear()
assert len(agent.history()) == 1   # history() returns a copy
agent.clear_history()
assert len(agent.history()) == 0
print("✅ SimpleAgent (run / add_tool / history / clear_history)")

print("\nSimple agent complete! Section 6 begins.")
